# Drift Gate — should the retraining job run?
First task of the retraining job. Checks the quality monitor's `_drift_metrics` table with the same
logic as `resources/feature_drift.alert.yml` (BASELINE drift, KS / chi-squared p-value < 0.05, features only)
over the last `drift_lookback_days` (default 7 — the job runs weekly).

Sets task value `retrain` (true/false) and `reason`. The next task (`retrain_needed`, a condition task)
only lets `build_features` → `train` → `pyfunc_wrapper` → `deploy` run when `retrain` is true.

* `force_retrain=true` (job parameter) → retrain regardless of drift (manual runs)
* drift metrics table missing (monitor not refreshed yet) → no retrain, logged as a warning

In [0]:
from src.config import CFG
from src.utils.logger import get_logger

logger = get_logger("monitoring.check_drift")

dbutils.widgets.text("force_retrain", "false")
dbutils.widgets.text("drift_lookback_days", "7")
FORCE_RETRAIN = dbutils.widgets.get("force_retrain").strip().lower() == "true"
LOOKBACK_DAYS = int(dbutils.widgets.get("drift_lookback_days"))

# Same threshold + column exclusions as resources/feature_drift.alert.yml
P_VALUE_THRESHOLD = 0.05
NON_FEATURE_COLS = [
    "at_risk", ":table", "Salesforce_Id__c", "inference_id",
    "served_model", "model_version", "prediction", "prediction_probability",
]
DRIFT_TABLE = f"{CFG.fq('evaluation')}_drift_metrics"

logger.info(f"Drift table: {DRIFT_TABLE} | lookback: {LOOKBACK_DAYS} days | force_retrain: {FORCE_RETRAIN}")

In [0]:
def drifted_features():
    """Features with significant BASELINE drift in the lookback window; None if the table doesn't exist."""
    if not spark.catalog.tableExists(DRIFT_TABLE):
        return None
    excluded = ", ".join(f"'{c}'" for c in NON_FEATURE_COLS)
    return spark.sql(f"""
        SELECT column_name,
               MIN(LEAST(COALESCE(ks_test.pvalue, 1.0), COALESCE(chi_squared_test.pvalue, 1.0))) AS min_pvalue
        FROM {DRIFT_TABLE}
        WHERE drift_type = 'BASELINE'
          AND slice_key IS NULL
          AND column_name NOT IN ({excluded})
          AND window.start >= current_date() - INTERVAL {LOOKBACK_DAYS} DAYS
          AND (ks_test.pvalue < {P_VALUE_THRESHOLD} OR chi_squared_test.pvalue < {P_VALUE_THRESHOLD})
        GROUP BY column_name
        ORDER BY min_pvalue
    """).collect()


if FORCE_RETRAIN:
    retrain, reason = True, "force_retrain=true"
else:
    rows = drifted_features()
    if rows is None:
        retrain, reason = False, f"{DRIFT_TABLE} not found — monitor has not refreshed yet"
        logger.warning(reason)
    elif rows:
        top = ", ".join(f"{r.column_name} (p={r.min_pvalue:.4f})" for r in rows[:5])
        retrain, reason = True, f"drift in {len(rows)} feature(s): {top}"
    else:
        retrain, reason = False, f"no feature drift (p < {P_VALUE_THRESHOLD}) in the last {LOOKBACK_DAYS} days"

logger.info(f"retrain={retrain} — {reason}")

# Read by the condition task: {{tasks.check_drift.values.retrain}}
dbutils.jobs.taskValues.set(key="retrain", value=retrain)
dbutils.jobs.taskValues.set(key="reason", value=reason)